## workflow outline

relevant website: https://medium.com/@v31u/building-an-ai-agent-that-turns-news-into-trading-signals-eb9e898e178c

### fetch news node
- prompt: "get news from the web"
- defined output: dataframe with textual features for each artcicle
    - ex: company, general sentiment

### analytics node
- prompt: "you are an experienced financial analyst. make evidence based decisions to predict the following features based on the the article content."
- defined output: news effectiveness on OPTIONS price (low, med, high), issue category? (constrain to market, supply chain, competetor etc.)
    - define a key trading signal

## VLLM model for finance 

https://huggingface.co/fastino/Fastino-Nemotron-3.5-Lightning-Finance

In [1]:
from vllm import LLM, SamplingParams

model_id = "fastino/Fastino-Nemotron-3.5-Lightning-Finance"

llm = LLM(
    model=model_id,
    trust_remote_code=True,
    dtype="bfloat16",
    max_model_len=4096,
)

outputs = llm.generate(
    ["test, say hello!"],
    SamplingParams(temperature=0.0, max_tokens=512),
)
print(outputs[0].outputs[0].text)




ModuleNotFoundError: No module named 'vllm'

## workflow

In [ ]:
import os
import getpass
import json 
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, MessagesState, START, END
from peewee import State

# prompts

fetch_news_prompt = f"get news related to this stock from the web"

def parse_json_file(file_path):

    """
    Opens a JSON file and parses it into a Python object.
    """
    file_path = f"/Users/joshuafeld/{file_path}"
    # 1. Verify the file exists before opening
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"No file found at: {file_path}")
        
    # 2. Open and safely parse the JSON data
    with open(file_path) as file:
        data = json.load(file)
        return data


if path:
    key = parse_json_file(path)["ica_key"]
    else: key="NONE"

llm = ChatOpenAI(model="gpt-5.6-luna-dzus", api_key = key)

# define nodes

def fetch_news(state: MessagesState):
    """first node to fetch relevant news articles"""
    return {"messages": [{"role": "user", "content": f"{fetch_news_prompt}"}]}

def create_graph(state: MessagesState):
    """second node to invoke the model"""
    messages = state["messages"]

    response = llm.invoke(messages)
    
    return {"messages": [response]} 

# create graph and register nodes

builder = StateGraph(MessagesState)

builder.add_node("fetch_news_node", fetch_news)
builder.add_node("llm_node", create_graph)

# define the flow (EDGES)

builder.add_edge(START, "fetch_news_node")
builder.add_edge("fetch_news_node", "llm_node")
builder.add_edge("llm_node", END)

graph = builder.compile()


In [ ]:
from typing import cast
initial_message = {"messages": [
    {
        "role": "user",
        "content": "tell me about aapl stock"
    }
]}

final = graph.invoke(cast(MessagesState, initial_message))